# Process Generation and Emissions Outputs for GM-SEUS Arrays
NOTE: Run after getGeneration, getGridEmissions, and getUpstreamEmissions. Needs `h3` (hexagon maps) in the BigPanel environment.

This script combines the outputs of the three modules into one long table (one row per array and operating year) and makes the summary figures with a shared figure theme.

**Combined long table** (`GMSEUS_GenEmissions_Annual_Long_<weather source>`): one row per array and operating year (operating year 1 = installation year).

| Group | Columns | Unit | From |
|---|---|---|---|
| Keys | `arrayID`, `instYr`, `opYr`, `calYr` | | all |
| Array | `STATEFP`, `capMWDC`, `pvMount`, `moduleFamily` | MW DC | GM-SEUS, getGeneration, getUpstreamEmissions |
| Generation | `MWh` | MWh | getGeneration |
| Grid rates | `rateAvgKgMWh`, `rateMarginalLRKgMWh`, `rateMarginalSRKgMWh`, `rateSource`, `rateSourceMarginal` | kg CO2e/MWh | getGridEmissions |
| Avoided emissions | `tCO2eAvoidedAvg`, `tCO2eAvoidedMarginalLR`, `tCO2eAvoidedMarginalSR` | t CO2e | getGridEmissions |
| Upstream emissions | `tCO2eUpstreamLow`, `tCO2eUpstreamCentral`, `tCO2eUpstreamHigh` (in the installation-year row, `opYr` = 1; 0 in other rows) | t CO2e | getUpstreamEmissions |
| Net | `tCO2eNetAvg`, `tCO2eNetMarginalLR`, `tCO2eNetMarginalSR` = avoided - central upstream | t CO2e | computed here |

**Sign convention**: avoided and upstream emissions are both positive numbers; net = avoided - upstream, so a positive net is a carbon benefit. In maps, blue is a carbon gain (net avoided) and red is a carbon loss, matching the land-use carbon figures.

**Versions** (definitions in getGridEmissions): **Average** = eGRID then Cambium average rate (attributional); **Marginal long-run** = AVERT then Cambium LRMER (primary marginal, lower bound); **Marginal short-run** = AVERT then Cambium SRMER (upper bound).

**Figures** (`Figures/genEmissions`, PNG at 600 dpi and vector PDF; widths follow Nature: 89 mm single, 183 mm double column)
1. Fleet time series: annual generation, annual avoided and upstream emissions, cumulative net.
2. Maps of lifetime net avoided emissions per H3 hexagon for each version, and carbon payback time.
3. Lifetime carbon budget by version: upstream, avoided (historical and future), net.
4. Emission intensities: grid rate by year, upstream factor by installation year, carbon payback distribution by mount.

# Import Libraries and Variables

In [ ]:
# Import libraries
import os
import h3
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from shapely.geometry import Polygon

In [ ]:
# Set folder paths
wd = r'S:\Users\stidjaco\R_files\BigPanel'
downloaded_path = os.path.join(wd, r'Data\Downloaded')
derived_path = os.path.join(wd, r'Data\Derived')
genEmissions_path = os.path.join(derived_path, r'genEmissions')
figure_path = os.path.join(wd, r'Figures\genEmissions')
genWeatherSource = 'pvgis' # which getGeneration / getGridEmissions run to use

# Module outputs (inputs here)
genLong_path = os.path.join(derived_path, r'generation\GMSEUS_Generation_Annual_Long_' + genWeatherSource + '.csv')
genInputs_path = os.path.join(derived_path, r'generation\GMSEUS_Generation_ModelInputs_' + genWeatherSource + '.csv')
offsetLong_path = os.path.join(derived_path, r'carbonOffset\GMSEUS_GridOffset_Annual_Long_' + genWeatherSource + '.csv')
upstream_path = os.path.join(derived_path, r'upstreamEmissions\GMSEUS_UpstreamEmissions.csv')

# GM-SEUS arrays (location, state) and boundaries for maps
gmseusArraysPath = os.path.join(downloaded_path, r'SolarDB\GMSEUS_v2_1\SHP\GMSEUS_Arrays_Final_2025_v2_1.shp')
countiesPath = os.path.join(downloaded_path, r'TIGER_Counties\tl_2019_us_county.shp') # dissolved to states

# Output path
combinedLong_path = os.path.join(genEmissions_path, r'GMSEUS_GenEmissions_Annual_Long_' + genWeatherSource + '.csv')

# Make folders if they do not exist
os.makedirs(genEmissions_path, exist_ok=True)
os.makedirs(figure_path, exist_ok=True)

# Settings
arrayIDcol = 'arrayID'
firstProjectedYear = 2025 # first Cambium (projected) year, for marking historical vs projected
h3Res = 4 # H3 resolution for maps (res 4: ~1,770 km2 hexagons, ~45 km across)
mapCRS = 'EPSG:5070' # CONUS Albers equal area
nonConusFips = ['02', '15', '60', '66', '69', '72', '78']

# Figure Theme
All figures use the shared theme in `utils/figureTheme.py` (also used by getGeneration, getGridEmissions, and getUpstreamEmissions): Arial, 7 pt text, thin axes with outward ticks, no top or right spines, no legend frames, bold lowercase panel letters, Nature column widths. Colors: carbon loss in reds and carbon gain in blues (binned, matching the land-use carbon maps), the three grid versions in a grey-to-blue sequence, upstream emissions in red, mounts in light grey / slate.

In [ ]:
# Shared figure theme (utils/figureTheme.py): style, colors, and helpers used by every figure in this project
import figureTheme as ft
ft.apply()
from figureTheme import mm, figWidth, versionColors, versionLabels, upstreamColor, upstreamDark, mountColors, mountLabels, textGrey, lineGrey, neutralColor, panelLabel, binnedDiverging, binnedBar, fmtNum

# Wrappers that fill in this script's settings
def markProjected(ax, **kw): ft.markProjected(ax, year=firstProjectedYear, **kw)
def styleMap(ax): ft.styleMap(ax, states, conus)
def saveFig(fig, name, folder=None): ft.saveFig(fig, folder or figure_path, name)

# Load and Combine Module Outputs

In [ ]:
# Call module outputs
offsetLong = pd.read_csv(offsetLong_path) # generation (MWh) + grid rates and avoided emissions per array-year
genLong = pd.read_csv(genLong_path)
genInputs = pd.read_csv(genInputs_path, usecols=[arrayIDcol, 'pvMount'])
upstream = pd.read_csv(upstream_path).replace(-9999, np.nan)
arrays = gpd.read_file(gmseusArraysPath, ignore_geometry=True, columns=[arrayIDcol, 'STATEFP', 'latitude', 'longitude'])

# Check: generation in the grid offset table matches getGeneration
chk = genLong.merge(offsetLong[[arrayIDcol, 'opYr', 'MWh']], on=[arrayIDcol, 'opYr'], suffixes=('Gen', 'Grid'))
print('Rows: generation', len(genLong), '| grid offset', len(offsetLong), '| matched', len(chk), '| max MWh difference', round((chk['MWhGen'] - chk['MWhGrid']).abs().max(), 3))

In [ ]:
# Build the combined long table
combined = offsetLong.rename(columns={'rateKgMWh': 'rateAvgKgMWh', 'tCO2e': 'tCO2eAvoidedAvg', 'tCO2eMarginalLR': 'tCO2eAvoidedMarginalLR', 'tCO2eMarginalSR': 'tCO2eAvoidedMarginalSR'})
combined = combined.merge(arrays[[arrayIDcol, 'STATEFP']], on=arrayIDcol, how='left')
combined = combined.merge(upstream[[arrayIDcol, 'capMWDC', 'pvMount', 'moduleFamily', 'tCO2eUpstream_low', 'tCO2eUpstream_central', 'tCO2eUpstream_high']], on=arrayIDcol, how='left')

# Upstream emissions sit in the installation-year row (opYr 1), zero elsewhere
for b, B in zip(['low', 'central', 'high'], ['Low', 'Central', 'High']):
    combined['tCO2eUpstream' + B] = np.where(combined['opYr'] == 1, combined['tCO2eUpstream_' + b], 0.0)
combined = combined.drop(columns=['tCO2eUpstream_low', 'tCO2eUpstream_central', 'tCO2eUpstream_high'])

# Net = avoided - central upstream
versions = ['Avg', 'MarginalLR', 'MarginalSR']
for v in versions:
    combined['tCO2eNet' + v] = combined['tCO2eAvoided' + v] - combined['tCO2eUpstreamCentral']

# Column order
combined = combined[[arrayIDcol, 'instYr', 'opYr', 'calYr', 'STATEFP', 'capMWDC', 'pvMount', 'moduleFamily', 'MWh',
                     'rateAvgKgMWh', 'rateMarginalLRKgMWh', 'rateMarginalSRKgMWh', 'rateSource', 'rateSourceMarginal',
                     'tCO2eAvoidedAvg', 'tCO2eAvoidedMarginalLR', 'tCO2eAvoidedMarginalSR',
                     'tCO2eUpstreamLow', 'tCO2eUpstreamCentral', 'tCO2eUpstreamHigh',
                     'tCO2eNetAvg', 'tCO2eNetMarginalLR', 'tCO2eNetMarginalSR']]

# Checks: every array-year has upstream, totals add up
print('Arrays:', combined[arrayIDcol].nunique(), '| rows:', len(combined), '| rows missing upstream:', combined['tCO2eUpstreamCentral'].isna().sum())
assert np.isclose(combined['tCO2eUpstreamCentral'].sum(), upstream['tCO2eUpstream_central'].sum())
totals = pd.DataFrame({versionLabels[v]: {'avoided (Mt)': combined['tCO2eAvoided' + v].sum() / 1e6, 'upstream central (Mt)': combined['tCO2eUpstreamCentral'].sum() / 1e6,
                                          'net (Mt)': combined['tCO2eNet' + v].sum() / 1e6} for v in versions})
print('Generation (TWh):', round(combined['MWh'].sum() / 1e6, 1)); print(totals.round(1).to_string())

# Per-Array Summary and Map Inputs

In [ ]:
# Per-array lifetime totals and simple carbon payback (central upstream / operating-year-1 avoided)
yr1 = combined[combined['opYr'] == 1].set_index(arrayIDcol)
arraySum = combined.groupby(arrayIDcol).agg(MWh=('MWh', 'sum'), **{'avoided' + v: ('tCO2eAvoided' + v, 'sum') for v in versions}, **{'net' + v: ('tCO2eNet' + v, 'sum') for v in versions},
                                            upstream=('tCO2eUpstreamCentral', 'sum'), upstreamLow=('tCO2eUpstreamLow', 'sum'), upstreamHigh=('tCO2eUpstreamHigh', 'sum'))
arraySum = arraySum.join(yr1[['instYr', 'capMWDC', 'pvMount', 'moduleFamily']])
for v in versions:
    arraySum['payback' + v] = arraySum['upstream'] / yr1['tCO2eAvoided' + v]

# Assign each array to an H3 hexagon and sum per hexagon
arraySum = arraySum.join(arrays.set_index(arrayIDcol)[['latitude', 'longitude', 'STATEFP']])
arraySum['h3'] = [h3.latlng_to_cell(la, lo, h3Res) for la, lo in zip(arraySum['latitude'], arraySum['longitude'])]
hexSum = arraySum.groupby('h3').agg(nArrays=('MWh', 'size'), capMWDC=('capMWDC', 'sum'), upstream=('upstream', 'sum'),
                                    **{'net' + v: ('net' + v, 'sum') for v in versions}, **{'avoidedYr1' + v: ('payback' + v, lambda s: np.nan) for v in versions})
for v in versions: # capacity-level payback per hexagon = hexagon upstream / hexagon year-1 avoided
    hexSum['payback' + v] = hexSum['upstream'] / yr1.join(arraySum['h3']).groupby('h3')['tCO2eAvoided' + v].sum()
hexSum = hexSum.drop(columns=['avoidedYr1' + v for v in versions])
hexGDF = gpd.GeoDataFrame(hexSum, geometry=[Polygon([(lo, la) for la, lo in h3.cell_to_boundary(c)]) for c in hexSum.index], crs='EPSG:4326').to_crs(mapCRS)

# States (dissolved from TIGER counties) and CONUS outline, in the map CRS
counties = gpd.read_file(countiesPath, columns=['STATEFP'])
states = counties[~counties['STATEFP'].isin(nonConusFips)].dissolve(by='STATEFP').to_crs(mapCRS)
conus = gpd.GeoDataFrame(geometry=[states.union_all() if hasattr(states, 'union_all') else states.unary_union], crs=mapCRS)
print('Hexagons with arrays:', len(hexGDF), '| arrays outside CONUS states (not mapped):', (~arraySum['STATEFP'].isin(states.index)).sum())

# Figure 1: Fleet Time Series
a, annual generation; b, annual avoided emissions for each version, with central upstream emissions as negative bars (low-high whiskers) in the installation year; c, cumulative net avoided emissions.

In [ ]:
yearly = combined.groupby('calYr').agg(MWh=('MWh', 'sum'), up=('tCO2eUpstreamCentral', 'sum'), upLow=('tCO2eUpstreamLow', 'sum'), upHigh=('tCO2eUpstreamHigh', 'sum'),
                                       **{'av' + v: ('tCO2eAvoided' + v, 'sum') for v in versions}, **{'net' + v: ('tCO2eNet' + v, 'sum') for v in versions})
yrs = yearly.index.values
fig, axes = plt.subplots(3, 1, figsize=(figWidth['single'], 120 * mm), sharex=True, gridspec_kw={'height_ratios': [0.8, 1.2, 1], 'hspace': 0.18})

# a: generation (historical darker, projected lighter)
ax = axes[0]
hist = yrs < firstProjectedYear
ax.bar(yrs[hist], yearly['MWh'][hist] / 1e6, width=0.85, color='#7D8B99', lw=0)
ax.bar(yrs[~hist], yearly['MWh'][~hist] / 1e6, width=0.85, color='#C9D0D6', lw=0)
ax.set_ylabel('Generation\n(TWh yr$^{-1}$)'); markProjected(ax); panelLabel(ax, 'a', x=-0.2)

# b: avoided emissions by version, marginal range shaded, upstream as negative bars
ax = axes[1]
ax.fill_between(yrs, yearly['avMarginalLR'] / 1e6, yearly['avMarginalSR'] / 1e6, color=versionColors['MarginalLR'], alpha=0.15, lw=0, label='Marginal range')
for v in versions:
    ax.plot(yrs, yearly['av' + v] / 1e6, color=versionColors[v], lw=1.1, label=versionLabels[v])
ax.bar(yrs, -yearly['up'] / 1e6, width=0.85, color=upstreamColor, lw=0, label='Upstream (central)')
ax.errorbar(yrs[yearly['up'] > 0], -yearly['up'][yearly['up'] > 0] / 1e6, yerr=[(yearly['upHigh'] - yearly['up'])[yearly['up'] > 0] / 1e6, (yearly['up'] - yearly['upLow'])[yearly['up'] > 0] / 1e6],
            fmt='none', ecolor='#5A0F0B', elinewidth=0.4, capsize=0)
ax.axhline(0, color='#222222', lw=0.6)
ax.set_ylabel('Avoided (+) and upstream (\u2212)\nemissions (Mt CO$_2$e yr$^{-1}$)'); markProjected(ax, labels=False)
ax.legend(loc='upper left', fontsize=6, bbox_to_anchor=(0.0, 0.92)); panelLabel(ax, 'b', x=-0.2)

# c: cumulative net
ax = axes[2]
for v in versions:
    ax.plot(yrs, yearly['net' + v].cumsum() / 1e9, color=versionColors[v], lw=1.1, label=versionLabels[v])
    ax.text(yrs[-1] + 0.6, yearly['net' + v].sum() / 1e9, fmtNum(yearly['net' + v].sum() / 1e9, 2), color=versionColors[v], fontsize=6, va='center')
ax.axhline(0, color='#222222', lw=0.6)
ax.set_ylabel('Cumulative net avoided\n(Gt CO$_2$e)'); ax.set_xlabel('Calendar year'); markProjected(ax, labels=False); panelLabel(ax, 'c', x=-0.2)
ax.legend(loc='upper left', fontsize=6, title='Net (avoided − upstream)', title_fontsize=6, alignment='left')
ax.set_xlim(2004.2, yrs.max() + 0.8) # few arrays before 2005 (their small totals are still included)
for a in axes: a.yaxis.set_label_coords(-0.12, 0.5)
saveFig(fig, 'fig1_fleetTimeseries')
plt.show()

# Figure 2: Maps of Lifetime Net Avoided Emissions and Carbon Payback
Lifetime (25-year) net avoided emissions per H3 hexagon (avoided - central upstream), for each version, binned on a symmetric log scale (blue = net carbon gain, red = net loss), and carbon payback time per hexagon (hexagon upstream / hexagon year-1 avoided emissions, average version).

In [ ]:
netEdges = [1, 10, 100, 1000, 10000] # kt CO2e per hexagon
cmapNet, normNet, edgesNet = binnedDiverging(netEdges)
paybackEdges = [0, 1, 2, 3, 5, 8, np.inf]
cmapPay = ListedColormap(['#14396F', '#2160AC', '#80B1E6', '#E3E2DD', '#EA877B', '#B02A22'])
normPay = BoundaryNorm(paybackEdges, cmapPay.N)

fig, axes = plt.subplots(2, 2, figsize=(figWidth['double'], 128 * mm), gridspec_kw={'hspace': 0.08, 'wspace': 0.02})
panels = [('netAvg', 'Average'), ('netMarginalLR', 'Marginal, long-run'), ('netMarginalSR', 'Marginal, short-run')]
for ax, (col, title), letter in zip(axes.flat[:3], panels, 'abc'):
    hexGDF.plot(ax=ax, column=hexGDF[col] / 1e3, cmap=cmapNet, norm=normNet, lw=0, zorder=2)
    styleMap(ax)
    ax.set_title(title, fontsize=8, pad=1)
    ax.text(0.5, 1.0, 'Net ' + ('+' if hexGDF[col].sum() > 0 else '') + fmtNum(hexGDF[col].sum() / 1e6, 0) + ' Mt CO$_2$e', transform=ax.transAxes, ha='center', va='top', fontsize=6.5, color=textGrey)
    panelLabel(ax, letter, x=0.0, y=1.0)
ax = axes.flat[3]
hexGDF.plot(ax=ax, column='paybackAvg', cmap=cmapPay, norm=normPay, lw=0, zorder=2)
styleMap(ax)
ax.set_title('Carbon payback time (average version)', fontsize=8, pad=1)
ax.text(0.5, 1.0, 'Fleet median ' + fmtNum(arraySum['paybackAvg'].median(), 1) + ' yr', transform=ax.transAxes, ha='center', va='top', fontsize=6.5, color=textGrey)
panelLabel(ax, 'd', x=0.0, y=1.0)

# Legends: binned net bar (left) and payback bar (right)
kFmt = ft.kFormat
binnedBar(fig, [0.07, 0.045, 0.40, 0.018], cmapNet, edgesNet, 'Lifetime net avoided emissions (kt CO$_2$e per hexagon; white: no arrays)', fmt=kFmt)
cax = fig.add_axes([0.58, 0.045, 0.34, 0.018])
for i in range(cmapPay.N): cax.add_patch(mpl.patches.Rectangle((i + 0.03, 0), 0.94, 1, color=cmapPay(i), lw=0))
cax.set_xlim(0, cmapPay.N); cax.set_ylim(0, 1); cax.set_yticks([]); [s.set_visible(False) for s in cax.spines.values()]
cax.set_xticks(np.arange(1, cmapPay.N)); cax.set_xticklabels([str(int(e)) for e in paybackEdges[1:-1]]); cax.tick_params(axis='x', length=0, pad=2, labelsize=6, colors=textGrey)
cax.set_xlabel('Carbon payback time (years)', fontsize=6.5, color=textGrey, labelpad=3)
cax.text(0, 1.15, 'Faster', transform=cax.transAxes, fontsize=6.5, fontweight='bold', ha='left', va='bottom'); cax.text(1, 1.15, 'Slower', transform=cax.transAxes, fontsize=6.5, fontweight='bold', ha='right', va='bottom')
saveFig(fig, 'fig2_netAvoidedMaps')
plt.show()

# Figure 3: Lifetime Carbon Budget by Version
Upstream emissions (negative, low-high whisker), avoided emissions split into historical (before 2025) and projected years, and the net (diamond, with whisker from the upstream range).

In [ ]:
histMask = combined['calYr'] < firstProjectedYear
up = combined['tCO2eUpstreamCentral'].sum() / 1e6
upLow, upHigh = combined['tCO2eUpstreamLow'].sum() / 1e6, combined['tCO2eUpstreamHigh'].sum() / 1e6
fig, ax = plt.subplots(figsize=(figWidth['single'], 62 * mm))
x = np.arange(len(versions)); w = 0.55
for i, v in enumerate(versions):
    avHist = combined.loc[histMask, 'tCO2eAvoided' + v].sum() / 1e6
    avFut = combined.loc[~histMask, 'tCO2eAvoided' + v].sum() / 1e6
    ax.bar(i, avHist, w, color=versionColors[v], edgecolor='#222222', lw=0.4)
    ax.bar(i, avFut, w, bottom=avHist, color=versionColors[v], alpha=0.45, edgecolor='#222222', lw=0.4)
    ax.bar(i, -up, w, color=upstreamColor, edgecolor='#222222', lw=0.4)
    ax.errorbar(i, -up, yerr=[[upHigh - up], [up - upLow]], color='#222222', lw=0.6, capsize=2, capthick=0.6)
    net = avHist + avFut - up
    ax.errorbar(i + w / 2 + 0.12, net, yerr=[[upHigh - up], [up - upLow]], fmt='D', ms=3.2, mfc='white', mec='#222222', mew=0.6, color='#222222', lw=0.6, capsize=1.5)
    ax.text(i + w / 2 + 0.2, net, fmtNum(net, 0), fontsize=6, va='center', ha='left', color='#222222')
ax.axhline(0, color='#222222', lw=0.6)
ax.set_xticks(x); ax.set_xticklabels([versionLabels[v].replace(', ', ',\n') for v in versions])
ax.set_ylabel('Lifetime emissions (Mt CO$_2$e)'); ax.set_xlim(-0.6, len(versions) - 0.1)
handles = [Patch(facecolor='#8A96A2', edgecolor='#222222', lw=0.4, label='Avoided, historical'), Patch(facecolor='#8A96A2', alpha=0.45, edgecolor='#222222', lw=0.4, label='Avoided, projected'),
           Patch(facecolor=upstreamColor, edgecolor='#222222', lw=0.4, label='Upstream'), Line2D([], [], marker='D', ls='none', ms=3.2, mfc='white', mec='#222222', mew=0.6, label='Net')]
ax.legend(handles=handles, loc='upper left', fontsize=6)
saveFig(fig, 'fig3_carbonBudget')
plt.show()

# Figure 4: Emission Intensities and Carbon Payback
a, fleet generation-weighted grid emission rate by year for each version; b, upstream emission factor by installation year (c-Si array median, low-high range; CdTe central); c, distribution of array carbon payback time by mount (average and long-run marginal versions).

In [ ]:
# Each panel is a function so the same drawing makes the multi-panel figure and the single-panel (presentation) figures
def drawGridRates(ax):
    rateYr = combined.assign(**{'w' + v: combined['MWh'] * combined[c] for v, c in zip(versions, ['rateAvgKgMWh', 'rateMarginalLRKgMWh', 'rateMarginalSRKgMWh'])}).groupby('calYr')[['MWh'] + ['w' + v for v in versions]].sum()
    rateYr = rateYr[rateYr['MWh'] > rateYr['MWh'].max() * 0.01] # years with at least 1% of peak generation
    for v in versions:
        ax.plot(rateYr.index, rateYr['w' + v] / rateYr['MWh'], color=versionColors[v], lw=1.1, label=versionLabels[v])
    ax.set_ylabel('Grid emission rate (kg CO$_2$e MWh$^{-1}$)'); ax.set_xlabel('Calendar year'); ax.set_ylim(bottom=0)
    markProjected(ax); ax.legend(loc='upper right', fontsize=6, bbox_to_anchor=(1.0, 0.9))

def drawUpstreamFactor(ax):
    csi = upstream[upstream['moduleFamily'] == 'c-si'].groupby('instYr')[['upstreamKgKW_low', 'upstreamKgKW_central', 'upstreamKgKW_high']].median()
    csi = csi[csi.index >= 2005]
    ax.fill_between(csi.index, csi['upstreamKgKW_low'] / 1e3, csi['upstreamKgKW_high'] / 1e3, color=upstreamColor, alpha=0.18, lw=0, label='c-Si, low\u2013high')
    ax.plot(csi.index, csi['upstreamKgKW_central'] / 1e3, color=upstreamColor, lw=1.1, label='c-Si, central')
    cdte = upstream[upstream['moduleFamily'] == 'cdte'].groupby('instYr')['upstreamKgKW_central'].median()
    ax.plot(cdte.index, cdte / 1e3, ls='none', marker='o', ms=2.2, mfc='white', mec=upstreamColor, mew=0.6, label='CdTe, central')
    ax.set_ylabel('Upstream emissions (t CO$_2$e kW$_{DC}^{-1}$)'); ax.set_xlabel('Installation year'); ax.set_ylim(bottom=0)
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.legend(loc='upper right', fontsize=6, title='Module + hardware,\nmedian array', title_fontsize=6, alignment='left')

def drawPayback(ax):
    mounts = ['fixed', 'single_axis', 'dual_axis']
    for j, v in enumerate(['Avg', 'MarginalLR']):
        data = [arraySum.loc[arraySum['pvMount'] == m, 'payback' + v].dropna().values for m in mounts]
        pos = np.arange(len(mounts)) + (j - 0.5) * 0.34
        bp = ax.boxplot(data, positions=pos, widths=0.28, whis=(5, 95), showfliers=False, patch_artist=True,
                        medianprops={'color': 'white', 'lw': 0.9}, whiskerprops={'lw': 0.6, 'color': versionColors[v]}, capprops={'lw': 0.6, 'color': versionColors[v]}, boxprops={'lw': 0})
        for patch in bp['boxes']: patch.set_facecolor(versionColors[v])
    ax.set_xticks(range(len(mounts))); ax.set_xticklabels([mountLabels[m] for m in mounts]); ax.set_xlim(-0.6, len(mounts) - 0.4)
    ax.set_ylabel('Carbon payback time (years)'); ax.set_ylim(bottom=0)
    ax.legend(handles=[Patch(facecolor=versionColors[v], label=versionLabels[v]) for v in ['Avg', 'MarginalLR']], title='Box: middle 50%, whiskers: 5th\u201395th percentile',
              title_fontsize=6, alignment='left', loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, fontsize=6, columnspacing=1.0)

fig = plt.figure(figsize=(figWidth['double'], 58 * mm))
gs = fig.add_gridspec(1, 3, wspace=0.42, width_ratios=[1.15, 1, 1])
for k, (draw, letter) in enumerate([(drawGridRates, 'a'), (drawUpstreamFactor, 'b'), (drawPayback, 'c')]):
    ax = fig.add_subplot(gs[k]); draw(ax); panelLabel(ax, letter, x=-0.22, y=1.1 if letter == 'c' else 1.04)
saveFig(fig, 'fig4_intensityPayback')
plt.show()

# Single-Panel Figures for Presentations
The same drawing functions as Figure 4, saved one panel per file (no panel letter) in `Figures/genEmissions/panels`, for slides.

In [ ]:
panelFolder = os.path.join(figure_path, 'panels')
for draw, name in [(drawGridRates, 'panel_gridRates'), (drawUpstreamFactor, 'panel_upstreamFactor'), (drawPayback, 'panel_payback')]:
    fig, ax = plt.subplots(figsize=(58 * mm, 60 * mm)) # small physical size so text stays legible when scaled up on a slide
    draw(ax)
    saveFig(fig, name, folder=panelFolder)
    plt.close(fig)

# Export

In [ ]:
# Export the combined long table
combined.round({'capMWDC': 4, 'MWh': 3, 'rateAvgKgMWh': 2, 'rateMarginalLRKgMWh': 2, 'rateMarginalSRKgMWh': 2,
                'tCO2eAvoidedAvg': 4, 'tCO2eAvoidedMarginalLR': 4, 'tCO2eAvoidedMarginalSR': 4,
                'tCO2eUpstreamLow': 4, 'tCO2eUpstreamCentral': 4, 'tCO2eUpstreamHigh': 4,
                'tCO2eNetAvg': 4, 'tCO2eNetMarginalLR': 4, 'tCO2eNetMarginalSR': 4}).to_csv(combinedLong_path, index=False)
print('Exported:', combinedLong_path)